# Klasifikasi Jeruk Lemon & Jeruk Purut (GLCM & OpenCV - Edisi Visual Close-Up)
**Tugas Penalaran Pola / Pengolahan Citra Digital**

Notebook ini menggunakan sampel foto **Close-Up Fokus Buah** (menghindari gangguan latar belakang meja):
- Sampel Jeruk Lemon: `20261007_210402(0).jpg`
- Sampel Jeruk Purut: `20261007_210135.jpg`

Dilengkapi dengan seluruh bukti visual tahapan ekstraksi fitur & klasifikasi.

**Urutan eksekusi (versi GLCM + K-NN):** Sel 1 → 1B → 2A → 2B → 2C → 2D → 2D-B → 3 → 3B → 4 → 4B → 5 (K-NN) → 6B (validasi silang) → 7 → 7B. Sel SVM sudah dihapus.

In [ ]:
# ===== SEL 1: MOUNT GOOGLE DRIVE & IMPORT LIBRARY =====
from google.colab import drive
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from skimage.feature import graycomatrix, graycoprops
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Hubungkan Google Drive
drive.mount('/content/drive')

# Path dataset jeruk di Google Drive Anda
DATASET_PATH = '/content/drive/MyDrive/Data Pepo Jeruk'

if os.path.exists(DATASET_PATH):
    print('✅ Folder dataset BERHASIL ditemukan! Subfolder/kelas:', os.listdir(DATASET_PATH))
else:
    print('❌ Folder belum ditemukan! Periksa kembali nama folder di MyDrive.')


In [ ]:
# ===== SEL 1B: UPLOAD FOTO DEMO (dipakai oleh Sel 2A, 2B, 2C, dan Sel 7) =====
# Ganti foto = jalankan ulang sel ini, lalu jalankan ulang Sel 2A, 2B, 2C, dan 7.
# Foto yang diunggah masuk ke /content (BUKAN ke Drive), jadi tidak ikut masuk dataset latih.
from google.colab import files

print('Upload 1 foto jeruk untuk didemokan (Lemon atau Purut):')
_up = files.upload()
DEMO_PATH = list(_up.keys())[0]
print('✅ Foto demo:', DEMO_PATH)

# Foto pembanding dari dataset (dipakai di Sel 2B & 2C). Ganti nama file bila ingin foto tertentu.
def foto_pembanding(kelas, nama_file=None):
    folder = os.path.join(DATASET_PATH, kelas)
    if nama_file and os.path.exists(os.path.join(folder, nama_file)):
        return os.path.join(folder, nama_file)
    return os.path.join(folder, sorted(os.listdir(folder))[0])

REF_LEMON = foto_pembanding('Jeruk Lemon', '20261007_210402(0).jpg')
REF_PURUT = foto_pembanding('Jeruk Purut', '20261007_210135.jpg')
print('Pembanding Lemon :', os.path.basename(REF_LEMON))
print('Pembanding Purut :', os.path.basename(REF_PURUT))

In [ ]:
# ===== SEL 2A: BUKTI VISUAL 1 - KONVERSI RGB KE GRAYSCALE, MATRIKS PIKSEL & BUKTI HITUNGAN =====
# 1. Baca dengan OpenCV (hasil awal berformat BGR) lalu resize 150x150
img_bgr = cv2.imread(DEMO_PATH)
img_bgr_small = cv2.resize(img_bgr, (150, 150))
img_resized = cv2.cvtColor(img_bgr_small, cv2.COLOR_BGR2RGB)      # untuk ditampilkan (RGB)

# 2. Konversi ke Grayscale (0.299*R + 0.587*G + 0.114*B)
img_gray = cv2.cvtColor(img_bgr_small, cv2.COLOR_BGR2GRAY)

# 3. Visualisasi
plt.figure(figsize=(15, 4))
plt.subplot(1, 3, 1)
plt.imshow(img_resized)
plt.title('1. Citra Asli RGB (Foto Demo)\nUkuran: 150x150x3 lapis warna')
plt.axis('off')

plt.subplot(1, 3, 2)
plt.imshow(img_gray, cmap='gray')
plt.title('2. Hasil Grayscale OpenCV\nMatriks 150x150 (Nilai 0-255)')
plt.axis('off')

plt.subplot(1, 3, 3)
center_matrix = img_gray[65:73, 65:73]
sns.heatmap(center_matrix, annot=True, fmt='d', cmap='YlGnBu', cbar=False)
plt.title('3. Bukti Nilai Angka Piksel Riil\n(Cuplikan 8x8 Piksel Foto Demo)')
plt.xlabel('Kolom Piksel')
plt.ylabel('Baris Piksel')
plt.tight_layout()
plt.show()

# ---------- BUKTI HITUNGAN 1: RGB -> GRAYSCALE pada piksel (baris 65, kolom 65) ----------
R, G, B = [int(v) for v in img_resized[65, 65]]
gray_manual = 0.299 * R + 0.587 * G + 0.114 * B
print('=' * 60)
print('BUKTI HITUNGAN 1: RGB -> GRAYSCALE (piksel baris 65, kolom 65)')
print('=' * 60)
print(f'Nilai RGB asli  : R = {R}, G = {G}, B = {B}')
print(f'Gray = (0.299 x {R}) + (0.587 x {G}) + (0.114 x {B})')
print(f'     = {0.299*R:.3f} + {0.587*G:.3f} + {0.114*B:.3f}')
print(f'     = {gray_manual:.2f}  ->  dibulatkan {round(gray_manual)}')
print(f'Nilai dari OpenCV (img_gray[65,65]) : {img_gray[65, 65]}')

# ---------- BUKTI HITUNGAN 2: LAPLACIAN pada piksel (65,65) ----------
g = img_gray.astype(int)                  # int agar tidak overflow uint8
patch = g[64:67, 64:67]                   # blok 3x3 di sekitar piksel (65,65)
pusat = patch[1, 1]
atas, bawah, kiri, kanan = patch[0, 1], patch[2, 1], patch[1, 0], patch[1, 2]
lap_manual = atas + bawah + kiri + kanan - 4 * pusat
lap_cv2 = cv2.Laplacian(img_gray, cv2.CV_64F)[65, 65]

print()
print('=' * 60)
print('BUKTI HITUNGAN 2: FILTER LAPLACIAN (piksel baris 65, kolom 65)')
print('=' * 60)
print('Blok 3x3 (img_gray[64:67, 64:67]):')
print(patch)
print(f'Pusat = {pusat} | Atas = {atas}, Bawah = {bawah}, Kiri = {kiri}, Kanan = {kanan}')
print(f'ΔI = ({atas} + {bawah} + {kiri} + {kanan}) - 4 x {pusat}')
print(f'   = {atas + bawah + kiri + kanan} - {4 * pusat} = {lap_manual}')
print(f'Nilai dari cv2.Laplacian [65,65]    : {lap_cv2:.0f}')

In [ ]:
# ===== SEL 2B: BUKTI VISUAL 2 - PERBANDINGAN TEKSTUR (FOTO DEMO vs PEMBANDING LEMON & PURUT) =====
def baca_gray(path):
    return cv2.cvtColor(cv2.resize(cv2.imread(path), (150, 150)), cv2.COLOR_BGR2GRAY)

daftar = [('Foto Demo', DEMO_PATH), ('Pembanding Lemon', REF_LEMON), ('Pembanding Purut', REF_PURUT)]

plt.figure(figsize=(15, 8))
for k, (nama, path) in enumerate(daftar):
    gray = baca_gray(path)
    lap = np.abs(cv2.Laplacian(gray, cv2.CV_64F))
    plt.subplot(2, 3, k + 1)
    plt.imshow(gray, cmap='gray')
    plt.title(f'{nama} (Grayscale)')
    plt.axis('off')
    plt.subplot(2, 3, k + 4)
    plt.imshow(lap, cmap='inferno')
    plt.title(f'Peta Tekstur Laplacian\nRata-rata |ΔI| = {lap.mean():.2f}')
    plt.axis('off')
plt.tight_layout()
plt.show()
print('Makin besar rata-rata |ΔI| = makin kasar/berkerut. Bandingkan Foto Demo dengan kedua pembanding.')

In [ ]:
# ===== SEL 2C: BUKTI VISUAL 3 - HEATMAP MATRIKS GLCM (FOTO DEMO vs PEMBANDING) =====
# Revisi: seluruh matriks 256x256 ditampilkan, label sumbu = tingkat keabuan ASLI (0-255),
# dan skala warna (vmax) disamakan agar ketiga foto bisa dibandingkan secara adil.
daftar = [('Foto Demo', DEMO_PATH), ('Pembanding Lemon', REF_LEMON), ('Pembanding Purut', REF_PURUT)]

matriks = []
for nama, path in daftar:
    gray = baca_gray(path)
    glcm = graycomatrix(gray, distances=[1], angles=[0], levels=256, symmetric=True, normed=True)
    matriks.append(glcm[:, :, 0, 0])          # seluruh 256 x 256, tanpa dipotong

vmax_sama = max(m.max() for m in matriks)     # skala warna yang sama untuk ketiga foto

plt.figure(figsize=(18, 5.5))
for k, ((nama, path), m) in enumerate(zip(daftar, matriks)):
    plt.subplot(1, 3, k + 1)
    sns.heatmap(m, cmap='viridis', vmin=0, vmax=vmax_sama, cbar=True,
                xticklabels=32, yticklabels=32, cbar_kws={'label': 'P(i,j)'})
    plt.title(f'Heatmap GLCM - {nama}\nPuncak P(i,j) = {m.max():.4f}')
    plt.xlabel('Tingkat keabuan tetangga (j), 0-255')
    plt.ylabel('Tingkat keabuan acuan (i), 0-255')
plt.tight_layout()
plt.show()
print('Skala warna sama untuk ketiga gambar. Pita diagonal lebih lebar & puncak lebih rendah = tekstur lebih kasar.')

# ---------- BUKTI HITUNGAN 3: PROBABILITAS P(i,j) FOTO DEMO (d=1, theta=0) ----------
gray_demo = baca_gray(DEMO_PATH)
glcm_cnt = graycomatrix(gray_demo, distances=[1], angles=[0], levels=256, symmetric=True, normed=False)[:, :, 0, 0]
total = glcm_cnt.sum()
i0 = int(gray_demo[65, 65])
print('=' * 60)
print('BUKTI HITUNGAN 3: P(i,j) = jumlah pasangan (i,j) / total pasangan')
print('=' * 60)
print(f'Total pasangan = 150 x 149 x 2 = {150*149*2}  |  dari komputer: {int(total)}')
for j in [i0, i0 + 1, i0 + 2, min(i0 + 56, 255)]:
    n = int(glcm_cnt[i0, j])
    print(f'Pasangan ({i0},{j}) muncul {n:>5} kali -> P = {n}/{int(total)} = {n/total:.4f}')

In [ ]:
# ===== SEL 2D: PROSES EKSTRAKSI SEMUA FOTO & EXPORT CSV =====
data_features = []
labels = []
file_names = []

categories = [f for f in os.listdir(DATASET_PATH) if os.path.isdir(os.path.join(DATASET_PATH, f))]
print(f'Mengekstrak fitur dari {len(categories)} kelas: {categories}...')

for category in categories:
    category_path = os.path.join(DATASET_PATH, category)
    images = os.listdir(category_path)
    
    for img_name in images:
        img_path = os.path.join(category_path, img_name)
        img = cv2.imread(img_path)
        if img is None:
            continue
            
        img_resized = cv2.resize(img, (150, 150))
        gray = cv2.cvtColor(img_resized, cv2.COLOR_BGR2GRAY)
        
        # Matriks GLCM 4 arah sudut
        glcm = graycomatrix(gray, distances=[1], angles=[0, np.pi/4, np.pi/2, 3*np.pi/4], levels=256, symmetric=True, normed=True)
        
        contrast = graycoprops(glcm, 'contrast').mean()
        dissimilarity = graycoprops(glcm, 'dissimilarity').mean()
        homogeneity = graycoprops(glcm, 'homogeneity').mean()
        energy = graycoprops(glcm, 'energy').mean()
        correlation = graycoprops(glcm, 'correlation').mean()
        
        data_features.append([contrast, dissimilarity, homogeneity, energy, correlation])
        labels.append(category)
        file_names.append(img_name)

df = pd.DataFrame(data_features, columns=['Contrast', 'Dissimilarity', 'Homogeneity', 'Energy', 'Correlation'])
df.insert(0, 'Nama_File', file_names)
df['Label'] = labels

df.to_csv('hasil_ekstraksi_fitur_glcm.csv', index=False)
print('✅ Ekstraksi Fitur Selesai! File disimpan sebagai hasil_ekstraksi_fitur_glcm.csv')
print(f'Total sampel gambar yang berhasil diekstrak: {len(df)}')
df.head(10)


In [ ]:
# ===== SEL 2D-B: FITUR FOTO DEMO PER SUDUT (bukti rata-rata 4 sudut) =====
# Jalankan setelah Sel 2D. Tidak menghitung ulang 512 foto, hanya foto demo.
nama_fitur = ['Contrast', 'Dissimilarity', 'Homogeneity', 'Energy', 'Correlation']
sudut = {'0': 0, '45': np.pi / 4, '90': np.pi / 2, '135': 3 * np.pi / 4}

gray_demo = baca_gray(DEMO_PATH)
hasil = {}
for nama, a in sudut.items():
    g = graycomatrix(gray_demo, distances=[1], angles=[a], levels=256, symmetric=True, normed=True)
    hasil[nama] = [graycoprops(g, p.lower())[0, 0] for p in nama_fitur]
tabel_sudut = pd.DataFrame(hasil, index=nama_fitur).T
tabel_sudut.loc['rata-rata'] = tabel_sudut.mean()

print('Fitur foto demo per sudut (baris rata-rata = angka yang dipakai model, sama dengan Sel 7):')
print(tabel_sudut.round(4).to_string())

fig, axes = plt.subplots(1, 5, figsize=(17, 3.6))
for ax, f in zip(axes, nama_fitur):
    ax.bar(list(sudut.keys()), tabel_sudut.loc[list(sudut.keys()), f], color='#5E9B2F')
    ax.axhline(tabel_sudut.loc['rata-rata', f], color='#C98A12', linestyle='--', label='rata-rata')
    ax.set_title(f'{f}\nrata-rata = {tabel_sudut.loc["rata-rata", f]:.4f}', fontsize=10)
    ax.set_xlabel('Sudut (derajat)')
axes[0].legend(fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# ===== SEL 3: BUKTI VISUAL 4 - SEBARAN DATA & RATA-RATA FITUR =====
plt.figure(figsize=(15, 5))

# Plot 1: Scatter plot
plt.subplot(1, 3, 1)
sns.scatterplot(data=df, x='Contrast', y='Homogeneity', hue='Label', style='Label', s=80, palette='Set1')
plt.title('Pemisahan: Contrast vs Homogeneity')
plt.grid(True)

# Plot 2: Boxplot Contrast
plt.subplot(1, 3, 2)
sns.boxplot(data=df, x='Label', y='Contrast', hue='Label', palette='Set2', legend=False)
plt.title('Distribusi Nilai Contrast')
plt.grid(True)

# Plot 3: Bar Chart Perbandingan Rata-Rata Fitur Tekstur
plt.subplot(1, 3, 3)
mean_features = df.groupby('Label')[['Contrast', 'Dissimilarity', 'Homogeneity']].mean()
mean_features.T.plot(kind='bar', ax=plt.gca(), colormap='viridis')
plt.title('Rata-Rata Nilai Fitur Tekstur')
plt.ylabel('Nilai Rata-rata')
plt.xticks(rotation=0)
plt.grid(True, axis='y')

plt.tight_layout()
plt.show()


In [ ]:
# ===== SEL 3B: STATISTIK PASTI CONTRAST PER KELAS (pengganti angka perkiraan Sel 3) =====
# Jalankan setelah Sel 3.
stat = df.groupby('Label')['Contrast'].describe().round(2)
print(stat.to_string())

urut = sorted(df['Label'].unique())
iqr = {l: stat.loc[l, '75%'] - stat.loc[l, '25%'] for l in urut}
print()
for l in urut:
    print(f'IQR {l} = Q3 - Q1 = {stat.loc[l, "75%"]} - {stat.loc[l, "25%"]} = {iqr[l]:.2f}')
r_mean = stat.loc[urut[-1], 'mean'] / stat.loc[urut[0], 'mean']
mid = (stat.loc[urut[0], '50%'] + stat.loc[urut[-1], '50%']) / 2
print(f'Rasio rata-rata {urut[-1]} / {urut[0]} = {stat.loc[urut[-1], "mean"]} / {stat.loc[urut[0], "mean"]} = {r_mean:.2f}')
print(f'Titik tengah dua median = ({stat.loc[urut[0], "50%"]} + {stat.loc[urut[-1], "50%"]}) / 2 = {mid:.2f}')

print()
print('Rata-rata kelima fitur per kelas:')
print(df.groupby('Label')[['Contrast', 'Dissimilarity', 'Homogeneity', 'Energy', 'Correlation']].mean().round(4).to_string())

plt.figure(figsize=(7, 4.5))
sns.boxplot(data=df, x='Label', y='Contrast', order=urut, hue='Label', palette='Set2', legend=False)
for k, l in enumerate(urut):
    for nilai, nm in [(stat.loc[l, '25%'], 'Q1'), (stat.loc[l, '50%'], 'Median'), (stat.loc[l, '75%'], 'Q3')]:
        plt.text(k + 0.28, nilai, f'{nm} = {nilai:.1f}', va='center', fontsize=9)
plt.title('Boxplot Contrast dengan angka pasti')
plt.grid(True, axis='y')
plt.show()

In [ ]:
# ===== SEL 4: PEMBAGIAN DATA & STANDARISASI Z-SCORE =====
X = df[['Contrast', 'Dissimilarity', 'Homogeneity', 'Energy', 'Correlation']].values
y = df['Label'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f'Jumlah Data Latih (Train): {len(X_train)}')
print(f'Jumlah Data Uji (Test): {len(X_test)}')


In [ ]:
# ===== SEL 4B: MU DAN SIGMA PASTI (SCALER) & JUMLAH PER KELAS =====
# Jalankan setelah Sel 4.
nama_fitur = ['Contrast', 'Dissimilarity', 'Homogeneity', 'Energy', 'Correlation']
tabel_scaler = pd.DataFrame({'mu (rata-rata latih)': scaler.mean_, 'sigma (simpangan baku latih)': scaler.scale_}, index=nama_fitur)
print(tabel_scaler.round(4).to_string())

print()
print('Jumlah data latih per kelas:', pd.Series(y_train).value_counts().to_dict())
print('Jumlah data uji per kelas  :', pd.Series(y_test).value_counts().to_dict())

print()
print('Z dari rata-rata Contrast tiap kelas (intuisi: Lemon negatif, Purut positif):')
for l, m in df.groupby('Label')['Contrast'].mean().items():
    z = (m - scaler.mean_[0]) / scaler.scale_[0]
    print(f'  {l}: Z = ({m:.2f} - {scaler.mean_[0]:.4f}) / {scaler.scale_[0]:.4f} = {z:+.4f}')

In [ ]:
# ===== SEL 5: KLASIFIKASI K-NN (AKURASI 100%) =====
knn = KNeighborsClassifier(n_neighbors=3, metric='euclidean')
knn.fit(X_train_scaled, y_train)

y_pred_knn = knn.predict(X_test_scaled)
acc_knn = accuracy_score(y_test, y_pred_knn)

print('=== HASIL KLASIFIKASI K-NEAREST NEIGHBORS (K-NN) ===')
print(f'Akurasi Model K-NN: {acc_knn * 100:.2f}%\n')
print('Classification Report K-NN:')
print(classification_report(y_test, y_pred_knn))

cm_knn = confusion_matrix(y_test, y_pred_knn)
plt.figure(figsize=(5, 4))
sns.heatmap(cm_knn, annot=True, fmt='d', cmap='Blues', xticklabels=np.unique(y), yticklabels=np.unique(y))
plt.title('Confusion Matrix K-NN')
plt.xlabel('Prediksi')
plt.ylabel('Aktual (Sebenarnya)')
plt.show()


In [ ]:
# ===== SEL 6B: VALIDASI SILANG - FOTO MANA YANG SALAH & KENAPA =====
# Jalankan setelah Sel 6.
from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_val_predict
from sklearn.pipeline import make_pipeline

model_knn = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=3))

print('Akurasi tiap lipatan:')
for nama, model in [('K-NN', model_knn)]:
    for acak in [False, True]:
        cv = StratifiedKFold(n_splits=5, shuffle=acak, random_state=42 if acak else None)
        skor = cross_val_score(model, X, y, cv=cv)
        print(f'  {nama} | diacak={acak!s:<5} | {skor.round(4)} | rata-rata {skor.mean()*100:.2f}% | simpangan {skor.std()*100:.2f}%')

cv0 = StratifiedKFold(n_splits=5, shuffle=False)
pred = cross_val_predict(model_knn, X, y, cv=cv0)

print()
print('Isi tiap lipatan (K-NN, tanpa acak):')
akurasi_lipatan = []
for k, (tr, te) in enumerate(cv0.split(X, y), start=1):
    benar = int((pred[te] == y[te]).sum())
    akurasi_lipatan.append(benar / len(te))
    print(f'  Lipatan {k}: {len(te)} foto, benar {benar}, salah {len(te) - benar}')

salah = df[pred != y].copy()
print(f'\nTotal foto salah: {len(salah)}')
if len(salah) > 0:
    salah['Prediksi'] = pred[pred != y]
    print(salah[['Nama_File', 'Label', 'Prediksi', 'Contrast', 'Homogeneity']].round(4).to_string())
    print(f'\nDari {len(salah)} foto salah, yang Homogeneity > 0.3: {(salah["Homogeneity"] > 0.3).sum()}')
    print(f'Dari seluruh {len(df)} foto, yang Homogeneity > 0.3: {(df["Homogeneity"] > 0.3).sum()}')

fig, ax = plt.subplots(1, 2, figsize=(14, 4))
warna = ['#C0392B' if a < 0.9 else '#5E9B2F' for a in akurasi_lipatan]
ax[0].bar(range(1, 6), akurasi_lipatan, color=warna)
for i, a in enumerate(akurasi_lipatan, start=1):
    ax[0].text(i, a + 0.01, f'{a:.4f}', ha='center', fontsize=9)
ax[0].set_ylim(0, 1.1)
ax[0].set_xlabel('Lipatan')
ax[0].set_ylabel('Akurasi')
ax[0].set_title('Akurasi per lipatan (K-NN, tanpa acak)')
sns.scatterplot(data=df, x='Contrast', y='Homogeneity', color='lightgray', s=40, ax=ax[1], label='semua foto')
if len(salah) > 0:
    ax[1].scatter(salah['Contrast'], salah['Homogeneity'], color='#C0392B', s=70, label='foto yang salah')
ax[1].set_title('Posisi foto yang salah di sebaran fitur')
ax[1].legend()
plt.tight_layout()
plt.show()

if len(salah) > 0:
    n = min(6, len(salah))
    plt.figure(figsize=(3 * n, 3.4))
    for k in range(n):
        r = salah.iloc[k]
        im = cv2.imread(os.path.join(DATASET_PATH, r['Label'], r['Nama_File']))
        plt.subplot(1, n, k + 1)
        if im is not None:
            plt.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB))
        plt.title(f"{r['Nama_File']}\nasli: {r['Label']}\ntebak: {r['Prediksi']}", fontsize=8)
        plt.axis('off')
    plt.suptitle('Contoh foto yang salah ditebak', y=1.02)
    plt.show()

In [ ]:
# ===== SEL 7: UJI COBA CITRA BARU (memakai FOTO DEMO dari Sel 1B) =====
img = cv2.imread(DEMO_PATH)
img_resized = cv2.resize(img, (150, 150))
gray = cv2.cvtColor(img_resized, cv2.COLOR_BGR2GRAY)

glcm = graycomatrix(gray, distances=[1], angles=[0, np.pi/4, np.pi/2, 3*np.pi/4], levels=256, symmetric=True, normed=True)
c = graycoprops(glcm, 'contrast').mean()
d = graycoprops(glcm, 'dissimilarity').mean()
h = graycoprops(glcm, 'homogeneity').mean()
e = graycoprops(glcm, 'energy').mean()
corr = graycoprops(glcm, 'correlation').mean()

fitur_sample = scaler.transform([[c, d, h, e, corr]])
hasil_prediksi = knn.predict(fitur_sample)[0]

plt.figure(figsize=(4, 4))
plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
plt.title(f'Hasil Prediksi: {hasil_prediksi.upper()}', fontsize=14, color='green', fontweight='bold')
plt.axis('off')
plt.show()

nama_fitur = ['Contrast', 'Dissimilarity', 'Homogeneity', 'Energy', 'Correlation']
nilai = [c, d, h, e, corr]
print('Nilai Fitur Tekstur (GLCM) -> Z-Score (dari scaler data latih):')
for n, v, mu, sd, z in zip(nama_fitur, nilai, scaler.mean_, scaler.scale_, fitur_sample[0]):
    print(f'{n:<14}: X = {v:.4f} | mu = {mu:.4f} | sigma = {sd:.4f} | Z = ({v:.4f}-{mu:.4f})/{sd:.4f} = {z:+.4f}')

# ---------- BUKTI HITUNGAN K-NN: 3 tetangga terdekat ----------
jarak, idx = knn.kneighbors(fitur_sample, n_neighbors=3)
print()
print('3 tetangga terdekat (jarak Euclidean pada ruang Z-Score):')
for r, (dj, ix) in enumerate(zip(jarak[0], idx[0]), start=1):
    print(f'  Tetangga {r}: {y_train[ix]:<12} d = {dj:.4f}')
suara = pd.Series(y_train[idx[0]]).value_counts()
print('Voting:', ' | '.join(f'{k}: {v} suara' for k, v in suara.items()), '->', hasil_prediksi)

In [ ]:
# ===== SEL 7B: TIGA TETANGGA K-NN - VEKTOR Z, JARAK, DAN FOTONYA =====
# Jalankan setelah Sel 7.
# Indeks asli data latih (pembagian sama persis dengan Sel 4)
idx_latih, idx_uji = train_test_split(np.arange(len(X)), test_size=0.2, random_state=42, stratify=y)
assert (y[idx_latih] == y_train).all(), 'Pembagian tidak sama dengan Sel 4'

jarak, idx = knn.kneighbors(fitur_sample, n_neighbors=3)
print('Z foto demo:', fitur_sample[0].round(4))
tetangga = []
for r, (dj, ix) in enumerate(zip(jarak[0], idx[0]), start=1):
    zt = X_train_scaled[ix]
    baris = df.iloc[idx_latih[ix]]
    nm, lb = baris['Nama_File'], baris['Label']
    kuadrat = (fitur_sample[0] - zt) ** 2
    tetangga.append((nm, lb, dj))
    print(f'\nTetangga {r}: {nm} | {lb}')
    print('   Z tetangga      =', zt.round(4))
    print('   selisih kuadrat =', kuadrat.round(6))
    print(f'   jumlah = {kuadrat.sum():.6f}  ->  d = akar({kuadrat.sum():.6f}) = {dj:.4f}')
print('\nFoto demo :', os.path.basename(DEMO_PATH))

plt.figure(figsize=(14, 4))
plt.subplot(1, 4, 1)
plt.imshow(cv2.cvtColor(cv2.imread(DEMO_PATH), cv2.COLOR_BGR2RGB))
plt.title('Foto demo', fontsize=10)
plt.axis('off')
for k, (nm, lb, dj) in enumerate(tetangga, start=2):
    plt.subplot(1, 4, k)
    im = cv2.imread(os.path.join(DATASET_PATH, lb, nm))
    if im is not None:
        plt.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB))
    plt.title(f'Tetangga {k - 1}: {lb}\n{nm}\nd = {dj:.4f}', fontsize=9)
    plt.axis('off')
plt.tight_layout()
plt.show()